# About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [43]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd

In [80]:
emlap_catalogue = google_conf.setup(sheet_url="https://docs.google.com/spreadsheets/d/1bkHHTYc86K2IuEXqfYfkDNt5LovtvCU3gvqHIbVio88/edit?usp=sharing", service_account_path="../../../ServiceAccountsKey.json")


# Get the data and transpose
emlap_metadata_raw = google_conf.get_as_dataframe(emlap_catalogue.worksheet("Copy_of_Catalogue_08_04_2025"), row=1, include_index=False)
emlap_metadata_raw.set_index("author_working", inplace=True)
emlap_metadata_raw.index.name = None
emlap_metadata= emlap_metadata_raw.T
emlap_metadata.index = emlap_metadata.index.astype(str)
emlap_metadata.head(5)

,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,author_name,...,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments,OTHER,filename,NaN
"Augurello, Chrysopoeia",100001,True,True,713324,NaN,NaN,True,NaN,True,"Augurelli, Giovanni Aurelio",...,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,Noscemus,Unknown,NaN,Noscemus Wiki,Soranzo 2019,The 1518 Basel version is also in Noscemus,NaN,"Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Ven...",NaN
"Pseudo-Lull, Secretis",100002,True,False,NaN,NaN,NaN,True,NaN,True,Pseudo-Lull,...,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,Hirsch 1950,NaN,"There is a prior, 1514 edition of De secretis ...",NaN,Pseudo-Lull1518_De_secretis_naturae_MDZ.pdf,NaN
"Pantheus, Ars Transmutatione",100003,True,False,NaN,NaN,NaN,True,NaN,True,"Panteo, Giovanni Agostino",...,NaN,GB,BL,NaN,NaN,NaN,This book was first published in 1518 with an ...,NaN,Pantheus1518_Ars_Transmutationis_Metallicae_BL...,NaN
"Pantheus, Commentarium",100004,True,False,NaN,NaN,NaN,True,NaN,True,"Panteo, Giovanni Agostino",...,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MSB,NaN,NaN,NaN,This 1519 book is catalogued wrongly by many l...,NaN,Pantheus1519_Commentarium_Transmutationis_Meta...,NaN
"Pantheus, Voarchadumia",100005,True,False,NaN,NaN,NaN,True,NaN,True,"Panteo, Giovanni Agostino",...,NaN,ONB,ONB,NaN,NaN,NaN,Dedicated to Leonellus Marquis of Estense,NaN,Pantheus1530_Voarchadumia_ONB.pdf,NaN


In [81]:
emlap_metadata[emlap_metadata["author_name"].str.startswith("Dorn", na=False)]

,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,author_name,...,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments,OTHER,filename,NaN
"Dorn, Artificii chymistici",100020,True,False,NaN,NaN,NaN,False,NaN,True,"Dorn, Gerard",...,NaN,MDZ,MBS,NaN,NaN,NaN,NaN,NaN,Dorn1569_Artificii_chymistici_MDZ_MBS.pdf,NaN
"Dorn, Clavis",100021,True,False,NaN,NaN,NaN,False,NaN,True,"Dorn, Gerard",...,NaN,ONB,ONB,NaN,NaN,NaN,NaN,NaN,Dorn1567_Clavis_totius_philosophiae_chymistica...,NaN
"Dorn, Lapis metaphysicus",100035,True,False,NaN,NaN,NaN,True,NaN,True,"Dorn, Gerard",...,NaN,MDZ,MBS,NaN,NaN,NaN,NaN,NaN,Dorn1570_Lapis_metaphysicus_MDZ_MBS_pdf.pdf,NaN
"Dorn, Theophrasti Germani",100044,True,False,NaN,NaN,NaN,True,NaN,True,"Dorn, Gerard",...,NaN,MDZ,MBS,NaN,NaN,NaN,NaN,NaN,Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS...,NaN
"Dorn, De naturae luce physica",100045,True,False,NaN,NaN,NaN,False,NaN,True,"Dorn, Gerard",...,NaN,MDZ,MBS,NaN,NaN,NaN,NaN,NaN,Dorn1583_De_Naturae_luce_physica_MDZ_MBS.pdf,NaN
"Dorn, Commentaria",100046,True,False,NaN,NaN,NaN,True,NaN,True,"Dorn, Gerard",...,NaN,MDZ,MBS,NaN,NaN,NaN,NaN,NaN,Dorn1584_Commentaria_in_Archidoxorum_MDZ_MBS.pdf,NaN


In [82]:
filename_id_dict = dict(zip(emlap_metadata["filename"], emlap_metadata["No."]))

For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [47]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

Tomela contains tuned latin preprocessing pipeline relying on spaCy and latinCy. You can check the pipeline as here:

In [48]:
tomela.nlp.pipeline

[('source_tracker', <function __main__.source_tracker(doc)>),
 ('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x71371a27cbf0>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x71371a27cef0>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x71371a27d1f0>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x71371a27d130>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x71371a27d070>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x71371bec0a50>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x71371bec0890>)]

In [49]:
tomela.nlp.max_length = 4000000

In [50]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(', '(', 'PUNCT')
('lib', 'liber', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [51]:
source_path = "/srv/data/tome/tome-corpus/emlap_annotated_textblocks/"
len(os.listdir(source_path))

152

In [52]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['Albertus1569_De_concordantia_Hippocraticorum_et_Paracelsistarum_MDZ_MBS.json',
 'Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS.json',
 'Anon1550_De_alchemia_opuscula_MDZ_MBS.json',
 'Anon1550_Rosarium_philosophorum_Erara.json',
 'Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.json',
 'Auriferae_artisI1572_MBZ_Augsburg.json',
 'Barnaud1599_Quadriga_aurifera_IA_Madrid.json',
 'Bodenstein1559_Isagoge_MDZ_MBS.json',
 'Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 'Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json',
 'Claveus1598_Apologia_crysopoeiae_MDZ_MBS.json',
 'De_alchemia1541_MDZ_MBS.json',
 'Dorn1567_Clavis_totius_philosophiae_chymisticae_ONB_pdf.json',
 'Dorn1569_Artificii_chymistici_MDZ_MBS.json',
 'Dorn1570_Lapis_metaphysicus_MDZ_MBS_pdf.json',
 'Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS.json',
 'Dorn1581_Fasciculus_Paracelsicae_MDZ_MBS.json',
 'Dorn1583_De_Naturae_luce_physica_MDZ_MBS.json',
 'Dorn1584_Commentaria_in_Archidoxorum_MDZ_MBS.json',
 'DuChes

In [83]:
[(f, filename_id_dict.get(f.replace(".json", ".pdf"), None)) for f in sorted(os.listdir(source_path)) if "_params" not in f]


[('Albertus1569_De_concordantia_Hippocraticorum_et_Paracelsistarum_MDZ_MBS.json',
  100062),
 ('Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS.json', 100072),
 ('Anon1550_De_alchemia_opuscula_MDZ_MBS.json', 100022),
 ('Anon1550_Rosarium_philosophorum_Erara.json', 100007),
 ('Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.json', 100001),
 ('Auriferae_artisI1572_MBZ_Augsburg.json', 100038),
 ('Barnaud1599_Quadriga_aurifera_IA_Madrid.json', 100050),
 ('Bodenstein1559_Isagoge_MDZ_MBS.json', 100017),
 ('Bonus1546_Pretiosa_Margarita_Novella_ONB.json', 100016),
 ('Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json', 100010),
 ('Claveus1598_Apologia_crysopoeiae_MDZ_MBS.json', 100056),
 ('De_alchemia1541_MDZ_MBS.json', 100011),
 ('Dorn1567_Clavis_totius_philosophiae_chymisticae_ONB_pdf.json', 100021),
 ('Dorn1569_Artificii_chymistici_MDZ_MBS.json', 100020),
 ('Dorn1570_Lapis_metaphysicus_MDZ_MBS_pdf.json', 100035),
 ('Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS.json', 100044),


In [84]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
files_processed = pd.DataFrame(files_overview)
files_processed

,filename,pages_n,chars_n
0,Moffett_De_iure_et_praestantia_MDZ_MBS.json,115,120620
1,Toxites1567_Spongia_stibii_MDZ_MBS.json,21,14543
2,Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json,405,340776
3,Pantheus1518_Ars_Transmutationis_Metallicae_BL...,53,49495
4,Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_...,69,105063
...,...,...,...
71,Paracelsus1553_Labyrinthus_medicorum_errantium...,95,173322
72,Fanianus1576_De_arte_metallicae_MDZ_MBS.json,129,96521
73,Pseudo-Democritus1572_Ars_magna.json,153,174854
74,Severinus1571_Idea_medicinae_MDZ_MBS.json,449,39747


In [56]:
google_conf.set_with_dataframe(emlap_catalogue.add_worksheet("files_processed", 1,1), files_processed, include_index=False)


# Develop and test with one example test

In [57]:
filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [58]:
len(textblocks)

93

In [59]:
textblocks[30][:10]

[{'coordinates': [165.1199951171875,
   40.31997299194336,
   427.239990234375,
   48.62395477294922],
  'text': '14\nRESPONSIO\n',
  'tag': 'header'},
 {'coordinates': [165.1199951171875,
   69.11996459960938,
   512.6112670898438,
   73.91996002197266],
  'text': 'calculo aut renum tartaro, tanta vi pro¬\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   93.3840103149414,
   515.9405517578125,
   98.30400848388672],
  'text': 'desse diceres? Intelligo, Confugeres\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   116.66397857666016,
   512.2813720703125,
   121.58397674560547],
  'text': 'ad sacram asinorum anchoram, nem¬\n',
  'tag': 'text'},
 {'coordinates': [168.72000122070312,
   140.87997436523438,
   509.6862487792969,
   145.6799774169922],
  'text': 'pe proprietatum occultarum: quod ta¬\n',
  'tag': 'text'},
 {'coordinates': [169.1999969482422,
   165.11996459960938,
   515.7952880859375,
   169.9199676513672],
  'text': 'men ipso sale fieri, qui illos r

In [60]:

# Modify the token extensions for simpler output
if not Token.has_extension("pages"):
    Token.set_extension("pages", default=None)
if not Token.has_extension("textblocks"):
    Token.set_extension("textblocks", default=None)
if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


def process_textblocks(textblocks):
    full_text = ""
    char_to_source = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "text":
                start_idx = len(full_text)
                text = tomela.text_cleaner(tb["text"])

                for char_idx in range(len(text)):
                    char_to_source[start_idx + char_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx
                    }

                full_text +=  text

    return full_text, char_to_source


@Language.component("source_tracker")
def source_tracker(doc):
    if doc._.char_to_source is not None:
        for token in doc:
            # Get the character span of the entire token
            token_char_range = range(token.idx, token.idx + len(token.text))

            pages = set()
            textblocks = set()

            for char_idx in token_char_range:
                if char_idx in doc._.char_to_source:
                    source_info = doc._.char_to_source[char_idx]
                    pages.add(source_info["page_idx"])
                    textblocks.add(source_info["textblock_idx"])

            token._.pages = sorted(list(pages))
            token._.textblocks = sorted(list(textblocks))
    return doc


# Add the custom component to your existing pipeline if not already added
if "source_tracker" not in tomela.nlp.pipe_names:
    tomela.nlp.add_pipe("source_tracker", before="senter")


def process_with_source_tracking(textblocks, nlp):
    full_text, char_to_source = process_textblocks(textblocks)
    # Create the doc with the text
    doc = nlp.make_doc(full_text)
    # Set the char_to_source before running the pipeline
    doc._.char_to_source = char_to_source
    # Process the doc through each pipeline component
    for name, proc in nlp.pipeline:
        doc = proc(doc)
    return doc

In [61]:
textblocks[21:22]

[[{'coordinates': [146.63999938964844,
    46.55996322631836,
    435.6400146484375,
    51.599952697753906],
   'text': '5\nAD AVBERTVM.\n',
   'tag': 'header'},
  {'coordinates': [88.55999755859375,
    74.66397857666016,
    435.73681640625,
    79.58397674560547],
   'text': 'naturae iuuandum robur, & aduersus af¬\n',
   'tag': 'text'},
  {'coordinates': [91.19999694824219,
    98.66397857666016,
    444.1780700683594,
    103.58397674560547],
   'text': 'fectus melancholicos, ad exolutum ven¬\n',
   'tag': 'text'},
  {'coordinates': [88.80000305175781,
    122.87997436523438,
    438.64984130859375,
    127.67996978759766],
   'text': 'triculum; ad cardiacos, & praeter ratio¬\n',
   'tag': 'text'},
  {'coordinates': [90.72000122070312,
    146.87997436523438,
    441.6309814453125,
    151.6799774169922],
   'text': 'nem moestos efficax remedium. Certe\n',
   'tag': 'text'},
  {'coordinates': [89.27999877929688,
    172.31997680664062,
    441.3201904296875,
    177.11997985839844

In [62]:
doc = process_with_source_tracking(textblocks[21:22], tomela.nlp)
doc

naturae iuuandum robur, & aduersus affectus melancholicos, ad exolutum uentriculum; ad cardiacos, & praeter rationem moestos efficax remedium. Certe in ipsius essentia, quam in tuo auro foliato, multo maiorem facultatem inesse merito credideris. Dabis & illud, mi Auberte, in eo purissimo, uim illam occultarum proprietatum maiorem esse, quam in tuis iusculis cum auro coctis. Nec tamen, puto, credes (hoc enim nimis esset absurdum) aurum, quod ne ignis quidem ardore torreri absumiue potest"(Uni enim (ut scribit Poeta) nil deperit auro Igne, uelut solum consumit nulla uetustas, Ac neque rubigo, aut aerugo conficit ulla: Cuncta adeo firmis illic compagibus haerent)" a natiuo calore decoqui aut deuinci ita posse, quin cor, integra remanente illius substantia, ipso corroborari quodammodo queat: quum sit haec Philosophorum sententia, Terram uidelicet omnem esse mortuam, & spiritus rerum in corporibus solos agere posse. Caeterum Laudanum ipsum quanuis opiaticum, non ita tamen conuitiis est 

In [63]:
doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), t._.pages, t._.textblocks) for t in sent]) for sent in doc.sents]
sent_data_updated = []
for n_sent, sent_data in enumerate(doc_sentdata):
    sent_data_updated.append((filename, n_sent, sent_data[0], sent_data[1]))

In [64]:
sent_data_updated

[('DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json',
  0,
  'naturae iuuandum robur, & aduersus affectus melancholicos, ad exolutum uentriculum;',
  [('naturae', 'natura', 'NOUN', (0, 7), [0], [1]),
   ('iuuandum', 'iuuo', 'VERB', (8, 16), [0], [1]),
   ('robur', 'robur', 'NOUN', (17, 22), [0], [1]),
   (',', ',', 'PUNCT', (22, 23), [0], [1]),
   ('&', '&', 'PUNCT', (24, 25), [0], [1]),
   ('aduersus', 'aduersus', 'ADP', (26, 34), [0], [1]),
   ('affectus', 'affectus', 'NOUN', (35, 43), [0], [1, 2]),
   ('melancholicos', 'melancholicus', 'ADJ', (44, 57), [0], [2]),
   (',', ',', 'PUNCT', (57, 58), [0], [2]),
   ('ad', 'ad', 'ADP', (59, 61), [0], [2]),
   ('exolutum', 'exoluo', 'VERB', (62, 70), [0], [2]),
   ('uentriculum', 'uentriculus', 'ADJ', (71, 82), [0], [2, 3]),
   (';', ';', 'PUNCT', (82, 83), [0], [3])]),
 ('DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json',
  1,
  'ad cardiacos, & praeter rationem moestos efficax remedium.',
  [('ad', 'ad', 'ADP', (0, 2), [0], [3]),
   ('car

In [85]:
target_path = "/srv/data/tome/tome-corpus/sents_data_id_jsons_v3-0/"
try:
    os.mkdir(target_path)
except:
    pass

In [91]:
filename_id_dict.items()

dict_items([('Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.pdf', 100001), ('Pseudo-Lull1518_De_secretis_naturae_MDZ.pdf', 100002), ('Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.pdf', 100003), ('Pantheus1519_Commentarium_Transmutationis_Metallicae_MDZ.pdf', 100004), ('Pantheus1530_Voarchadumia_ONB.pdf', 100005), ('Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.pdf', 100006), ('Anon1550_Rosarium_philosophorum_Erara.pdf', 100007), ('Severinus1572_Epistola_MBZ_MBS.pdf', 100008), ('Vegius1518_Inter_inferiora_corpora_disputatio_ONB.pdf', 100009), ('Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.pdf', 100010), ('De_alchemia1541_MDZ_MBS.pdf', 100011), ('Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.pdf', 100012), ('Ulstad1525_Coelum_philosophorum_IA_BIUSP_pdf.pdf', 100013), ('Toxites1567_Spongia_stibii_MDZ_MBS.pdf', 100014), ('Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.pdf', 100015), ('Bonus1546_Pretiosa_Margarita_Novella_ONB.pdf', 100016), ('Bodenste

In [94]:
%%time
source_path = "/srv/data/tome/tome-corpus/emlap_annotated_textblocks/"
for filename, id in filename_id_dict.items():
        try:
                if str(id) + ".json " not in os.listdir(target_path):
                    filename = filename.replace(".pdf", ".json")
                    filepath = os.path.join(source_path, filename.replace(".pdf", ".json"))
                    with open(filepath, 'r', encoding='utf-8') as f:
                            textblocks_pages = json.load(f)
                    print("currently processing: ", filename)
                    doc = process_with_source_tracking(textblocks_pages, tomela.nlp)
                    doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), t._.pages, t._.textblocks) for t in sent]) for sent in doc.sents]
                    sent_data_updated = []
                    for n_sent, sent_data in enumerate(doc_sentdata):
                            sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
                    with open(target_path + str(id) + ".json", "w") as f:
                            json.dump(sent_data_updated, f)
        except:
            print("failed with file: ", id, filename)
            pass

currently processing:  Augurello,_Giovanni_Aurelio_-_Chrysopoeia__Venice_1515_pdf.json
currently processing:  Pseudo-Lull1518_De_secretis_naturae_MDZ.json
currently processing:  Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json
currently processing:  Pantheus1519_Commentarium_Transmutationis_Metallicae_MDZ.json
currently processing:  Pantheus1530_Voarchadumia_ONB.json
currently processing:  Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json
currently processing:  Anon1550_Rosarium_philosophorum_Erara.json
currently processing:  Severinus1572_Epistola_MBZ_MBS.json
currently processing:  Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json
currently processing:  Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json
currently processing:  De_alchemia1541_MDZ_MBS.json
currently processing:  Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json
currently processing:  Ulstad1525_Coelum_philosophorum_IA_BIUSP_pdf.json
currently processing:  Toxites1567_Spongia_stibii_MDZ_MBS.json
curren

In [104]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

['100044.json',
 '100034.json',
 '100014.json',
 '100060.json',
 '100010.json',
 '100068.json',
 '100043.json',
 '100072.json',
 '100041.json',
 '100012.json']

In [105]:
len(fns_jsons)

75

In [99]:
sents_data = json.load(open(target_path + fns_jsons[20], "r"))
sents_data[100:103]

[[100042,
  100,
  'Quare certius atque perfectius alio quouis docere potest:',
  [['Quare', 'quare', 'ADV', [0, 5], [18], [7]],
   ['certius', 'certior', 'ADV', [6, 13], [18], [7]],
   ['atque', 'atque', 'CCONJ', [14, 19], [18], [7]],
   ['perfectius', 'perfectior', 'ADV', [20, 30], [18], [7]],
   ['alio', 'alius', 'DET', [31, 35], [18], [7]],
   ['quouis', 'quiuis', 'ADV', [36, 42], [18], [8]],
   ['docere', 'doceo', 'VERB', [43, 49], [18], [8]],
   ['potest', 'possum', 'VERB', [50, 56], [18], [8]],
   [':', ':', 'PUNCT', [56, 57], [18], [8]]]],
 [100042,
  101,
  'item absolute nos ab eo discere ualemus, quod loquitur inquiens:',
  [['item', 'item', 'ADV', [0, 4], [18], [8]],
   ['absolute', 'absolute', 'ADV', [5, 13], [18], [8]],
   ['nos', 'nos', 'PRON', [14, 17], [18], [8]],
   ['ab', 'ab', 'ADP', [18, 20], [18], [8]],
   ['eo', 'is', 'PRON', [21, 23], [18], [9]],
   ['discere', 'disco', 'VERB', [24, 31], [18], [9]],
   ['ualemus', 'ualeo', 'VERB', [32, 39], [18], [9]],
   [',', 

In [108]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v3-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [109]:
os.listdir(lemmatized_sents_path)

[]

In [110]:
for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    print(fn)
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position, t_pages, t_textblocks in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma)
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)

100044.json
100034.json
100014.json
100060.json
100010.json
100068.json
100043.json
100072.json
100041.json
100012.json
100025.json
100019.json
100047.json
100013.json
100028.json
100073.json
100070.json
100048.json
100038.json
100035.json
100042.json
100071.json
100053.json
100003.json
100059.json
100002.json
100049.json
100032.json
100052.json
100066.json
100045.json
100051.json
100001.json
100054.json
100065.json
100064.json
100004.json
100050.json
100046.json
100069.json
100058.json
100062.json
100033.json
100006.json
100029.json
100017.json
100061.json
100007.json
100074.json
100027.json
100063.json
100037.json
100067.json
100026.json
100023.json
100056.json
100024.json
100040.json
100021.json
100055.json
100005.json
100030.json
100018.json
100020.json
100009.json
100075.json
100008.json
100031.json
100015.json
100039.json
100036.json
100057.json
100016.json
100022.json
100011.json
